# Model Selection – Diabetes Risk (Improved)
This notebook selects and evaluates models to predict diabetes risk.

In [ ]:
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from xgboost import XGBClassifier

## Load training data

In [ ]:
# Cargar datos procesados de entrenamiento (no incluir test)
X_train = pd.read_parquet("../data/dataset/X_train.parquet")
y_train = pd.read_parquet("../data/dataset/y_train.parquet").squeeze()

print(X_train.shape, y_train.shape)
print(y_train.value_counts(normalize=True))

# Calcular scale_pos_weight para XGBoost
scale_pos_weight = y_train.value_counts()[0] / y_train.value_counts()[1]
scale_pos_weight

## Define models (baseline without heavy tuning)

In [ ]:
models = {
    "LogisticRegression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            penalty="l2",
            C=1.0,
            max_iter=5000,
            class_weight="balanced",
            random_state=42
        ))
    ]),

    "RandomForest": Pipeline([
        ("clf", RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ))
    ]),

    "HistGradientBoosting": Pipeline([
        ("clf", HistGradientBoostingClassifier(
            random_state=42
        ))
    ]),

    "XGBoost": Pipeline([
        ("clf", XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            eval_metric="logloss",
            scale_pos_weight=scale_pos_weight,
            random_state=42,
            n_jobs=-1
        ))
    ])
}

## Cross-validation configuration

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",  # PR AUC
    "recall": "recall",
    "precision": "precision",
    "f1": "f1"
}

## Model evaluation

In [ ]:
results = []

for name, model in models.items():
    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False
    )

    results.append({
        "model": name,
        "roc_auc_mean": scores["test_roc_auc"].mean(),
        "roc_auc_std": scores["test_roc_auc"].std(),
        "pr_auc_mean": scores["test_average_precision"].mean(),
        "pr_auc_std": scores["test_average_precision"].std(),
        "recall": scores["test_recall"].mean(),
        "precision": scores["test_precision"].mean(),
        "f1": scores["test_f1"].mean()
    })

results_df = pd.DataFrame(results).sort_values("pr_auc_mean", ascending=False)

print("\n=== Model comparison (sorted by PR-AUC) ===")
display(results_df)

## Selecting and saving the best model

In [ ]:
best_model_name = results_df.iloc[0]["model"]
print(f"\nBest model based on PR-AUC: {best_model_name}")

best_model = clone(models[best_model_name])
best_model.fit(X_train, y_train)

output_dir = "../data/models"
os.makedirs(output_dir, exist_ok=True)

model_path = os.path.join(output_dir, f"{best_model_name}.pkl")
joblib.dump(best_model, model_path)

print(f"Saved model: {best_model_name} → {model_path}")